In [1]:

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
import numpy as np
from collections import namedtuple
from IPython.display import HTML

In [2]:
from physics import *
from util import *

In [3]:
class Paddle(namedtuple('Paddle', ['rigidbody', 'collider', 'action_idx', 'pos_max', 'pos_min', 'v_max', 'team', 'turn'])):
    rigidbody: RigidBody
    collider: BoxCollider
    action_idx: jnp.array
    pos_max: jnp.array
    pos_min: jnp.array
    v_max: jnp.array
    turn : jnp.array # 0, 1
    team : jnp.array # 0, 1
    
    def update(self, config):
        next_rigidbody = self.rigidbody.update(config)
        next_rigidbody = next_rigidbody._replace(position=jnp.clip(next_rigidbody.position, self.pos_min, self.pos_max), velocity=jnp.clip(next_rigidbody.velocity, -self.v_max, self.v_max))
        
        return self._replace(rigidbody=next_rigidbody, collider=self.collider, action_idx=self.action_idx, pos_max=self.pos_max, pos_min=self.pos_min, v_max=self.v_max)
    
    def on_action (self, objects, info):
        action = info[self.action_idx] # 0: up, 1: down
        next_velocity = jnp.array([self.rigidbody.velocity[0], (action == 1) - 0.5]) * 2.0
        return self._replace(rigidbody=self.rigidbody._replace(velocity=next_velocity))
    
        
class Ball(namedtuple('Ball', ['rigidbody', 'collider', 'turn', 'touch_count', 'v_max'])):
    rigidbody: RigidBody
    collider: CircleCollider
    touch_count : jnp.array
    turn : jnp.array 
    v_max : jnp.array
    
    def update(self, config):
        rigidbody = self.rigidbody.update(config)
        rigidbody = rigidbody._replace(velocity=jnp.clip(rigidbody.velocity, -self.v_max, self.v_max))
        return self._replace(rigidbody=rigidbody)
    
    def on_collision(self, objects, is_collision, name):
        if 'paddle' in name:
            is_right_turn = objects[name].turn == self.turn
            notify(objects, 'ball_touch_paddle', (is_right_turn, objects[name].team, self.turn, is_collision))
            touch_count = self.touch_count + 1 * is_collision
            turn = (touch_count % 2)
            return self._replace(turn = turn, touch_count = touch_count)
        return self
    
        
class StaticBox(namedtuple('StaticBox', ['rigidbody', 'collider'])):
        rigidbody: RigidBody
        collider: BoxCollider
        def update(self, config):
            return self
        
class GoalLine(namedtuple('GoalLine', ['rigidbody', 'collider', 'score', 'team'])):
        rigidbody: RigidBody
        collider: BoxCollider
        score: jnp.array
        team : jnp.array
        def update(self, config):
            return self
        
        def on_collision(self, objects, is_collision, name):
            if 'ball' in name:
                notify(objects, 'goal', (self.team, is_collision))
                return self._replace(score=self.score + 1 * is_collision)
            else:
                return self
        
class GameManager(namedtuple('GameManager', ['reward', 'done', 'penalty'])):
    reward : jnp.array
    done : jnp.array
    penalty : jnp.array
    def update(self, config):
        return self._replace(reward = {'agent_0': jnp.array(0.0), 'agent_1': jnp.array(0.0), 'agent_2': jnp.array(0.0), 'agent_3': jnp.array(0.0)}, done = jnp.array(False))
    
    def on_ball_touch_paddle(self, objects, info):
        
        """
        Teams:
        - Left team: agent_0 (left paddle) and agent_1 (right paddle)
        - Right team: agent_2 (left paddle) and agent_3 (right paddle)
        """
        
        valid, team, agent_turn, is_collision = info
        reward0 = (-1.0 * (~valid) * (team == 0)) * is_collision * (agent_turn == 1)
        reward1 = (-1.0 * (~valid) * (team == 0)) * is_collision * (agent_turn == 0)
        reward2 = (-1.0 * (~valid) * (team == 1)) * is_collision * (agent_turn == 0)
        reward3 = (-1.0 * (~valid) * (team == 1)) * is_collision * (agent_turn == 1)
        
        
        reward = jax.tree.map(lambda x, y : x + y, self.reward, {'agent_0': reward0, 'agent_1': reward1, 'agent_2': reward2, 'agent_3': reward3})
        return self._replace(reward = reward)
    
    def on_goal(self, objects, info):
        team, is_collision = info
        left_team_reward = ((team == 0) * is_collision) * self.penalty
        right_team_reward = ((team == 1) * is_collision) * self.penalty
        done = is_collision
        
        reward = jax.tree.map(lambda x, y : x + y, self.reward, {'agent_0': left_team_reward, 'agent_1': left_team_reward, 'agent_2': right_team_reward, 'agent_3': right_team_reward})
        return self._replace(reward = reward, done = done | self.done)
    

In [4]:
from easydict import EasyDict
config = EasyDict()
config.dt = 0.02
config.percent = 0.5
config.slop = 0.01
config.restitution = 0.8

In [5]:
class Pong:
    """
    A Pong game environment with two teams, each controlling two paddles.
    
    The game consists of a ball bouncing between two sides of the screen.
    Each team tries to prevent the ball from passing their side while trying
    to make the ball pass the opponent's side.
    
    Teams:
    - Left team: agent_0 (left paddle) and agent_1 (right paddle)
    - Right team: agent_2 (left paddle) and agent_3 (right paddle)
    
    The agents receive rewards for successfully defending their side and
    scoring against the opponent. Similar to doubles tennis, agents receive
    negative rewards if they touch the ball when it's not their turn,
    encouraging proper coordination within each team.
    """
    def __init__(self, config):
        self.config = config
        self.update_sprites_target = ['ball', 'top_wall', 'bottom_wall', 'left_paddle0', 'left_paddle1', 'right_paddle0', 'right_paddle1', 'left_wall', 'right_wall']
        self.physics_sprites_target = ['ball', 'top_wall', 'bottom_wall', 'left_paddle0', 'left_paddle1', 'right_paddle0', 'right_paddle1', 'left_wall', 'right_wall']
    
    def get_obs(self, state):
        # left_team_obs = jnp.stack([state['ball'].rigidbody.position, state['ball'].rigidbody.velocity, state['left_paddle0'].rigidbody.position, state['left_paddle1'].rigidbody.position, state['right_paddle0'].rigidbody.position, state['right_paddle1'].rigidbody.position])
        
        
        # symmetry_vector = jnp.array([[-1.0, 1.0]])
        
        # right_team_obs = jnp.stack([state['ball'].rigidbody.position, state['ball'].rigidbody.velocity, state['right_paddle0'].rigidbody.position, state['right_paddle1'].rigidbody.position, state['left_paddle0'].rigidbody.position, state['left_paddle1'].rigidbody.position]) * symmetry_vector
        
        
        # return {
        #     'agent_0': left_team_obs,
        #     'agent_1' : left_team_obs,
        #     'agent_2': right_team_obs,
        #     'agent_3': right_team_obs
        # }

        return jnp.array(0)
    
    def reset(self, key, env_params = None):
        ball_init_velocity = jax.random.normal(key, (2, ))
        ball_init_velocity /= jnp.sqrt(jnp.square(ball_init_velocity).sum())
        ball = Ball(
        rigidbody=RigidBody(
            position=jnp.array([0.0, 0.0]),
            velocity=ball_init_velocity,
            mass=jnp.array(1.0),
            acceleration=jnp.array([0.0, 0.0])
        ),
        collider=CircleCollider(radius=0.1),
        turn=jax.random.randint(key, (1,), 0, 2)[0],
        touch_count=jnp.array(0),
        v_max=jnp.array(1.5)
        )
        
        
        static_mass = jnp.array(1e10)  
        
        top_wall = StaticBox(
            rigidbody=RigidBody(
                position=jnp.array([0.0, 2.5]),
                velocity=jnp.array([0.0, 0.0]),
                mass=static_mass,
                acceleration=jnp.array([0.0, 0.0])
            ),
            collider=BoxCollider(width=10.0, height=1.0)
        )
        bottom_wall = StaticBox(
            rigidbody=RigidBody(
                position=jnp.array([0.0, -2.5]),
                velocity=jnp.array([0.0, 0.0]),
                mass=static_mass,
                acceleration=jnp.array([0.0, 0.0])
            ),
            collider=BoxCollider(width=10.0, height=1.0)
        )
        
        left_wall = GoalLine(
            rigidbody=RigidBody(
                position=jnp.array([-5.0, 0.0]),
                velocity=jnp.array([0.0, 0.0]),
                mass=static_mass,
                acceleration=jnp.array([0.0, 0.0])
            ),
            collider=BoxCollider(width=1.0, height=4.0),
            score=jnp.array(0),
            team=jnp.array(0)
        )   
        
        right_wall = GoalLine(
            rigidbody=RigidBody(
                position=jnp.array([5.0, 0.0]),
                velocity=jnp.array([0.0, 0.0]),
                mass=static_mass,
                acceleration=jnp.array([0.0, 0.0]),
            ),
            collider=BoxCollider(width=1.0, height=4.0), 
            score=jnp.array(0),
            team=jnp.array(1)
        )
        left_paddle0 = Paddle(
            rigidbody=RigidBody(
                position=jnp.array([-2.5, 0.0]),
                velocity=jnp.array([0.0, 0.0]),
                mass=static_mass,
                acceleration=jnp.array([0.0, 0.0]),
            ),
            collider=BoxCollider(width=0.1, height=0.5),
            action_idx=jnp.array(0),
            pos_max=jnp.array([-2.5, 1.75]),
            pos_min=jnp.array([-2.5, -1.75]),
            v_max=jnp.array(1.0),
            team=jnp.array(0),
            turn=jnp.array(0)
        )

        left_paddle1 = Paddle(
            rigidbody=RigidBody(
                position=jnp.array([-3.0, 0.0]),
                velocity=jnp.array([0.0, 0.0]),
                mass=static_mass,
                acceleration=jnp.array([0.0, 0.0]),
            ),
            collider=BoxCollider(width=0.1, height=0.5),
            action_idx=jnp.array(1),
            pos_max=jnp.array([-3.0, 1.75]),
            pos_min=jnp.array([-3.0, -1.75]),
            v_max=jnp.array(1.0),
            team=jnp.array(0),
            turn=jnp.array(1)
        )
        
        
        right_paddle0 = Paddle(
            rigidbody=RigidBody(
                position=jnp.array([2.5, 0.0]),
                velocity=jnp.array([0.0, 0.0]),
                mass=static_mass,
                acceleration=jnp.array([0.0, 0.0]),
            ),
            collider=BoxCollider(width=0.1, height=0.5),
            action_idx=jnp.array(2),
            pos_max=jnp.array([2.5, 1.75]),
            pos_min=jnp.array([2.5, -1.75]),
            v_max=jnp.array(1.0),
            team=jnp.array(1),
            turn=jnp.array(0)
        )
        
        right_paddle1 = Paddle(
            rigidbody=RigidBody(
                position=jnp.array([3.0, 0.0]),
                velocity=jnp.array([0.0, 0.0]),
                mass=static_mass,
                acceleration=jnp.array([0.0, 0.0]),
            ),
            collider=BoxCollider(width=0.1, height=0.5),
            action_idx=jnp.array(3),
            pos_max=jnp.array([3.0, 1.75]),
            pos_min=jnp.array([3.0, -1.75]),
            v_max=jnp.array(1.0),
            team=jnp.array(1),
            turn=jnp.array(1)
        )

        
        
        game_manager = GameManager(
            reward={'agent_0': jnp.array(0.0), 'agent_1': jnp.array(0.0), 'agent_2': jnp.array(0.0), 'agent_3': jnp.array(0.0)},
            done=jnp.array(False),
            penalty=jnp.array(0.5)
        )
        # env_state = {
        #     'ball': ball,
        #     'top_wall': top_wall,
        #     'bottom_wall': bottom_wall,
        #     'left_paddle0': left_paddle0,
        #     'left_paddle1': left_paddle1,
        #     'right_paddle0': right_paddle0,
        #     'right_paddle1': right_paddle1,
        #     'left_wall': left_wall,
        #     'right_wall': right_wall,
        #     'game_manager': game_manager
        # }

        env_state = {
            'ball': ball,
            'top_wall': top_wall,
            'bottom_wall': bottom_wall,
            'axxxx': left_paddle0,
            'axxx2': left_paddle1,
            'axxx25': right_paddle0,
            'axxx1151': right_paddle1,
            'left_wall': left_wall,
            'right_wall': right_wall,
            'game_manager': game_manager
        }
        
        
        return self.get_obs(env_state), env_state
    
    def step(self, key_step, state, action, env_params):
        state = {
            sprite: state[sprite].update(self.config) for sprite in state.keys()
        }
        state = physics_step(config, state, [
            'ball',
            'top_wall',
            'bottom_wall',
            'axxxx',
            'axxx2',
            'axxx25',
            'axxx1151',
            'left_wall',
            'right_wall'
        ])
        
        state = notify(state, 'action', action)
        reward = state['game_manager'].reward
        done = state['game_manager'].done
        
        return self.get_obs(state), state, reward, done, {}
    

In [6]:
import pong as pong2
# from pong import Pong

pong = Pong(config)


In [7]:
obs, state = pong.reset(jax.random.key(0))


In [8]:
def get_action(obs, key):
    return jax.random.randint(key, (4,), 0, 2)



def step_body(carry, _):
    state, obs, key = carry
    key, next_key = jax.random.split(key)
    action = get_action(obs, key)
    obs, state, reward, done, info = pong.step(key, state, action, None)
    return (state, obs, next_key), (state, reward, done)




In [9]:
def get_action2(obs, key):
    return jax.random.randint(key, (4,1), 0, 2)



def step_body2(carry, _):
    state, obs, key = carry
    key, next_key = jax.random.split(key)
    action = get_action2(obs, key)
    obs, state, reward, done, info = pong2.step(key, state, {
        'agent_0': action[0],
        'agent_1': action[1],
        'agent_2': action[2],
        'agent_3': action[3]
    }, None)
    return (state, obs, next_key), (state, reward, done)




In [10]:
carry = (state, obs, jax.random.key(1))

In [11]:
carry, history = jax.lax.scan(step_body, carry, None, length=3000)

In [12]:
import pong as pong2
def visualize_frame(objects, ax):
    """현재 프레임의 모든 물체를 시각화합니다."""
    ax.clear()
    ax.set_xlim(-5, 5)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.set_title(f"{list(objects['game_manager'].reward.values())}")
    
    for obj in objects.values():
        if isinstance(obj, Ball) or isinstance(obj, pong2.Ball):
            # 공 그리기 - turn에 따라 색상 다르게
            ball_color = 'red' if obj.turn == 0 else 'blue'
            circle = plt.Circle(
                (obj.rigidbody.position[0], obj.rigidbody.position[1]),
                obj.collider.radius,
                color=ball_color,
                fill=True
            )
            ax.add_patch(circle)
        elif isinstance(obj, Paddle) or isinstance(obj, pong2.Paddle):
            # 패들 그리기 - turn에 따라 색상 다르게
            paddle_color = 'red' if obj.turn == 0 else 'blue'
            half_width = obj.collider.width / 2
            half_height = obj.collider.height / 2
            x = obj.rigidbody.position[0] - half_width
            y = obj.rigidbody.position[1] - half_height
            
            rectangle = patches.Rectangle(
                (x, y),
                obj.collider.width,
                obj.collider.height,
                color=paddle_color,
                fill=True
            )
            ax.add_patch(rectangle)
        elif isinstance(obj, GoalLine) or isinstance(obj, pong2.GoalLine):
            # 골라인 그리기 - 팀에 따라 색상 다르게
            goal_color = 'purple' if obj.team == 0 else 'green'
            half_width = obj.collider.width / 2
            half_height = obj.collider.height / 2
            x = obj.rigidbody.position[0] - half_width
            y = obj.rigidbody.position[1] - half_height
            
            rectangle = patches.Rectangle(
                (x, y),
                obj.collider.width,
                obj.collider.height,
                color=goal_color,
                fill=True,
                alpha=0.7
            )
            ax.add_patch(rectangle)
            ax.text(obj.rigidbody.position[0], obj.rigidbody.position[1], f'Score: {obj.score}', ha='center', va='center', color='white', fontweight='bold')
        elif isinstance(obj, StaticBox) or isinstance(obj, pong2.StaticBox):
            # 일반 벽 그리기
            half_width = obj.collider.width / 2
            half_height = obj.collider.height / 2
            x = obj.rigidbody.position[0] - half_width
            y = obj.rigidbody.position[1] - half_height
            
            rectangle = patches.Rectangle(
                (x, y),
                obj.collider.width,
                obj.collider.height,
                color='gray',
                fill=True
            )
            ax.add_patch(rectangle)
        
    return ax.patches

def create_animation(history, interval=50, save_gif=False, filename='simulation.gif', frame_skip=1):
    """시뮬레이션 결과의 애니메이션을 생성합니다."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # 프레임 스킵 적용
    total_frames = len(history['ball'].rigidbody.position)
    frames_to_show = range(0, total_frames, frame_skip)
    
    def update(frame_idx):
        frame = frames_to_show[frame_idx]
        return visualize_frame(jax.tree.map(lambda x: x[frame], history), ax)
    
    ani = FuncAnimation(
        fig, 
        update, 
        frames=len(frames_to_show),
        interval=interval,
        blit=True
    )
    
    # GIF로 저장
    if save_gif:
        ani.save(filename, writer='pillow', fps=1000//interval)
        print(f"Animation saved as {filename}")
    
    plt.close()  # 정적 그림 표시 방지
    return HTML(ani.to_jshtml())

def save_animation_as_gif(history, Box, filename='simulation.gif', interval=50, frame_skip=1):
    """시뮬레이션 결과를 GIF 파일로 저장합니다."""
    return create_animation(history, interval=interval, save_gif=True, filename=filename, frame_skip=frame_skip)

In [13]:
history[0]['game_manager'].done.mean()

Array(0.00066667, dtype=float32)

In [14]:

# 애니메이션 생성 및 표시
animation = create_animation(history[0], interval=10, save_gif=True, filename='simulation.gif', frame_skip=30)
animation

Animation saved as simulation.gif
